# 09 — Surface-Code Foundations

## Read before you begin

- Fowler, Mariantoni, Martinis, Cleland, [*"Surface codes: Towards practical large-scale quantum computation"*](https://arxiv.org/abs/1208.0928), Phys. Rev. A 86, 032324 (2012).
- Exact sections to read: **III. THE SURFACE CODE**, **IV. QUIESCENT STATE OF THE SURFACE CODE**, **V. SINGLE-QUBIT ERRORS**, **VI. LOGICAL OPERATORS**, **VII. ERROR DETECTION**.
- Concepts to extract: the two roles qubits play (data vs. measurement/ancilla) on a 2D lattice; how "measure-Z" and "measure-X" ancillas each stabilize a 4-qubit plaquette; $X$ boundaries vs. $Z$ boundaries and how logical operators become strings running between opposite boundaries; how the paper defines error chains and why minimum-weight matching is the natural decoding strategy (notebook 10 implements this).

## Learning objectives

Every code so far (repetition, Hamming, Shor, Steane) was a small, fixed block: adding more protection meant a fundamentally different code with more, and more complicated, non-local checks. The surface code takes a different approach: repeat one simple, *local* checking pattern over a bigger 2D grid. By the end of this notebook you should be able to:

1. identify the two qubit roles (data, measurement) on a surface-code lattice and explain what each measure-$Z$/measure-$X$ ancilla stabilizes;
2. explain what makes a check "$X$-type" or "$Z$-type" and verify it directly from a real circuit;
3. explain what a syndrome *defect* is, and why a single error typically triggers a *pair* of defects rather than one;
4. explain logical operators as strings of operators connecting opposite boundaries, and see how their length is set by the lattice size;
5. explain, concretely (not just by citation), why the surface code's distance scales with lattice size while a small block code's does not, and why that difference matters for large-scale error correction.

**Expected outputs:** a real distance-3 surface-code circuit (built by Stim's own generator, not hand-assembled), a labeled data/ancilla layout, and a distance-vs-qubit-count table showing the surface code's scaling behavior.

**Assumes:** notebooks 03–07 (stabilizers, syndromes, CSS codes, code distance). This notebook does not re-derive those definitions — it applies them to a new, 2D geometry.

## 1. Why not just build a bigger Steane code?

Every code from notebooks 04–07 was a **fixed block**: the $[7,4,3]$ Hamming code and the Steane code it produces are always distance 3, no matter how many of them you use — reaching distance 5 requires a genuinely *different*, more complicated code (a bigger, denser parity-check matrix, with checks that can span many, potentially far-apart, qubits). Section III of the reading solves this differently: define one simple, repeating pattern of **local** checks (each check only ever touches its 4 nearest neighbors) on a 2D grid, and get a *family* of codes, indexed by the grid size, whose distance grows with the grid — using the exact same kind of check throughout. This is the key practical reason surface codes, not bigger Steane-like codes, are the leading approach to large-scale hardware: real hardware qubits can only talk to their physical neighbors, and the surface code is built entirely from nearest-neighbor operations.

## 2. Data qubits and measurement (ancilla) qubits

Section III of the reading distinguishes two roles: **data qubits** (open circles, storing the encoded information) and **measurement qubits** (solid circles: "measure-$Z$" and "measure-$X$"), each coupled to its 4 neighboring data qubits. A measure-$Z$ qubit forces its 4 neighbors into an eigenstate of $Z_aZ_bZ_cZ_d$; a measure-$X$ qubit does the same for $X_aX_bX_cX_d$ — exactly notebook 05's CSS idea (separate families of $X$-type and $Z$-type checks), now arranged on a grid instead of built from a classical Hamming matrix.

Rather than hand-building this circuit (error-prone once you're placing dozens of qubits on a grid), we use Stim's own built-in surface-code circuit generator — the same tool notebook 10 will use for decoding.

In [1]:
import stim

circuit_d3 = stim.Circuit.generated("surface_code:rotated_memory_z", distance=3, rounds=1)
print(circuit_d3)

QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(1, 3) 8
QUBIT_COORDS(2, 2) 9
QUBIT_COORDS(3, 3) 10
QUBIT_COORDS(4, 2) 11
QUBIT_COORDS(5, 3) 12
QUBIT_COORDS(6, 2) 13
QUBIT_COORDS(0, 4) 14
QUBIT_COORDS(1, 5) 15
QUBIT_COORDS(2, 4) 16
QUBIT_COORDS(3, 5) 17
QUBIT_COORDS(4, 4) 18
QUBIT_COORDS(5, 5) 19
QUBIT_COORDS(4, 6) 25
R 1 3 5 8 10 12 15 17 19 2 9 11 13 14 16 18 25
TICK
H 2 11 16 25
TICK
CX 2 3 16 17 11 12 15 14 10 9 19 18
TICK
CX 2 1 16 15 11 10 8 14 3 9 12 18
TICK
CX 16 10 11 5 25 19 8 9 17 18 12 13
TICK
CX 16 8 11 3 25 17 1 9 10 18 5 13
TICK
H 2 11 16 25
TICK
MR 2 9 11 13 14 16 18 25
DETECTOR(0, 4, 0) rec[-4]
DETECTOR(2, 2, 0) rec[-7]
DETECTOR(4, 4, 0) rec[-2]
DETECTOR(6, 2, 0) rec[-5]
M 1 3 5 8 10 12 15 17 19
DETECTOR(0, 4, 1) rec[-3] rec[-6] rec[-13]
DETECTOR(2, 2, 1) rec[-5] rec[-6] rec[-8] rec[-9] rec[-16]
DETECTOR(4, 4, 1) rec[-1] rec[-2] rec[-4] rec[-5] rec[-11]
DETECTOR(6, 2, 1) rec[-4] rec[-7] rec[-14]
OBSERVABLE_INCLUDE(0) rec[

### Prediction 1

**Before running the next cell:** this is the smallest ("distance-3") surface code. Based on Section III's description (each data qubit touches up to 4 measurement qubits, each measurement qubit touches up to 4 data qubits, arranged on a square-ish grid), how many data qubits would you guess a distance-3 patch needs? Roughly how many measurement qubits?

In [2]:
def surface_code_layout(distance, rounds=1):
    """Classify a Stim-generated rotated surface-code circuit's qubits by role."""
    circuit = stim.Circuit.generated("surface_code:rotated_memory_z", distance=distance, rounds=rounds)
    coords = circuit.get_final_qubit_coordinates()
    data_qubits, ancilla_qubits, h_targeted = set(), set(), set()
    for instruction in circuit:
        name = instruction.name
        targets = {int(t.value) for t in instruction.targets_copy()}
        if name == "M":
            data_qubits |= targets
        elif name == "MR":
            ancilla_qubits |= targets
        elif name == "H":
            h_targeted |= targets
    x_ancillas = h_targeted & ancilla_qubits  # Hadamard-basis measurement -> measure-X
    z_ancillas = ancilla_qubits - x_ancillas
    return coords, data_qubits, x_ancillas, z_ancillas


coords, data_qubits, x_ancillas, z_ancillas = surface_code_layout(distance=3)
print(f"data qubits:        {len(data_qubits)}  -> {sorted(data_qubits)}")
print(f"measure-X ancillas: {len(x_ancillas)}  -> {sorted(x_ancillas)}")
print(f"measure-Z ancillas: {len(z_ancillas)}  -> {sorted(z_ancillas)}")
print(f"total checks (m):   {len(x_ancillas) + len(z_ancillas)}")
print(f"n - m = {len(data_qubits)} - {len(x_ancillas) + len(z_ancillas)} = {len(data_qubits) - len(x_ancillas) - len(z_ancillas)}  (matches k=1 logical qubit)")

data qubits:        9  -> [1, 3, 5, 8, 10, 12, 15, 17, 19]
measure-X ancillas: 4  -> [2, 11, 16, 25]
measure-Z ancillas: 4  -> [9, 13, 14, 18]
total checks (m):   8
n - m = 9 - 8 = 1  (matches k=1 logical qubit)


$n=9$ data qubits (a $3\times3$ patch — not a coincidence: $n = d^2$ for a distance-$d$ rotated surface code) and $m=8$ checks (4 of each type), giving $k = n - m = 1$ — the same $k=n-m$ counting rule from notebook 06, just with $n$ and $m$ both much larger relative to $k$ than any small block code needed.

## 3. Reading the grid layout

Every qubit's role can be plotted directly from its coordinates, giving the same picture as the reading's Fig. 1: data qubits and the two ancilla types interleaved on a checkerboard.

In [3]:
def print_grid(coords, data_qubits, x_ancillas, z_ancillas):
    by_xy = {}
    for q, xy in coords.items():
        x, y = xy
        if q in data_qubits:
            label = "D"
        elif q in x_ancillas:
            label = "X"
        elif q in z_ancillas:
            label = "Z"
        else:
            continue
        by_xy[(round(x), round(y))] = label

    xs = [xy[0] for xy in by_xy]
    ys = [xy[1] for xy in by_xy]
    for y in range(min(ys), max(ys) + 1):
        row = "".join(by_xy.get((x, y), ".") for x in range(min(xs), max(xs) + 1))
        print(row)


print_grid(coords, data_qubits, x_ancillas, z_ancillas)
print("\n(D = data qubit, X = measure-X ancilla, Z = measure-Z ancilla, . = no qubit there)")

..X....
.D.D.D.
..Z.X.Z
.D.D.D.
Z.X.Z..
.D.D.D.
....X..

(D = data qubit, X = measure-X ancilla, Z = measure-Z ancilla, . = no qubit there)


## 4. Syndrome defects

Section VII calls a measurement outcome that differs from what the noiseless circuit would deterministically produce a **defect** (Stim calls the same thing a `DETECTOR` firing). Section III notes each data qubit is coupled to *two* measure-$Z$ and *two* measure-$X$ ancillas — so an interior data qubit's error should trip a *pair* of same-type detectors, while a qubit sitting on the boundary (coupled to only one ancilla of that type) trips just one.

To inject a real, single, well-defined error we use Stim's `X_ERROR(1.0)` noise instruction (not a bare `X` gate — a bare gate is deterministic circuit structure, not noise, so `DETECTOR` would just silently treat it as part of the "expected" trajectory and never flag it).

In [4]:
def inject_single_error(circuit, qubit, kind="X"):
    """Return a copy of `circuit` with a certain X or Z error forced onto `qubit`, right after reset."""
    reset_index = next(i for i, instr in enumerate(circuit) if instr.name == "R")
    corrupted = circuit.copy()
    corrupted.insert(reset_index + 1, stim.CircuitInstruction(f"{kind}_ERROR", [qubit], [1.0]))
    return corrupted


base_circuit = stim.Circuit.generated("surface_code:rotated_memory_z", distance=3, rounds=1)

print(f"{'data qubit':>10} {'coordinate':>12} {'defects fired':>14}")
for q in sorted(data_qubits):
    corrupted = inject_single_error(base_circuit, q, kind="X")
    detection_events = corrupted.compile_detector_sampler().sample(shots=1)[0]
    fired = [i for i, bit in enumerate(detection_events) if bit]
    print(f"{q:>10} {str(coords[q]):>12} {str(fired):>14}")

data qubit   coordinate  defects fired
         1   [1.0, 1.0]            [1]
         3   [3.0, 1.0]            [1]
         5   [5.0, 1.0]            [3]
         8   [1.0, 3.0]         [0, 1]
        10   [3.0, 3.0]         [1, 2]
        12   [5.0, 3.0]         [2, 3]
        15   [1.0, 5.0]            [0]
        17   [3.0, 5.0]            [2]
        19   [5.0, 5.0]            [2]


### Prediction 2

**Before moving on:** based on the counts just printed, which data qubits are "interior" (2 defects) and which are on the boundary (1 defect)? Check your guess against the grid printed in Section 3. Now, given Section VII's description, if you injected *two* separate single-qubit errors on two data qubits that are far apart on the grid, how many defects would you expect in total, and what would happen if both errors landed on the very same qubit instead?

(Answer, briefly: two well-separated single-qubit errors should each create their own defect(s) independently, adding up — unless a defect from one happens to land on the same check as a defect from the other. Two $X$ errors on the *same* qubit cancel out entirely ($X^2=I$), leaving zero defects: an undetectable, but harmless, non-error.)

## 5. Logical operators as strings

Section VI defines the lattice's two edge types — **$X$ boundaries** and **$Z$ boundaries** — and constructs logical operators as strings of single-qubit operators that connect one boundary to its opposite. In the circuit Stim generated, that string is exactly what `OBSERVABLE_INCLUDE` names: the specific set of data-qubit measurement outcomes whose parity encodes the logical value.

In [5]:
for instruction in circuit_d3:
    if instruction.name == "OBSERVABLE_INCLUDE":
        targets = [t.value for t in instruction.targets_copy()]
        print(f"Logical observable involves {len(targets)} measurement record(s): {targets}")

Logical observable involves 3 measurement record(s): [-7, -8, -9]


The logical operator's weight (here, 3 data-qubit measurements) grows with the size of the patch — Section VI's "string spanning the lattice" is literally a path of length $d$ across the grid. This is the geometric reason the surface code's distance equals its lattice size $d$: shrinking a logical string below weight $d$ would require a shortcut through the lattice that the code's boundary structure doesn't allow.

## 6. Distance scales with the grid, not with a new code

### Prediction 3

**Before running the next cell:** based on $n = d^2$ (Section 2), how many data qubits should a distance-5 surface code need? A distance-7 code?

In [6]:
print(f"{'distance':>8} {'data qubits (n)':>16} {'checks (m)':>12} {'k = n - m':>10}")
for d in [3, 5, 7, 9]:
    coords_d, data_d, xa_d, za_d = surface_code_layout(distance=d)
    n, m = len(data_d), len(xa_d) + len(za_d)
    print(f"{d:>8} {n:>16} {m:>12} {n - m:>10}")

distance  data qubits (n)   checks (m)  k = n - m
       3                9            8          1
       5               25           24          1
       7               49           48          1
       9               81           80          1


Every one of these is built from the *exact same kind of check* — a 4-qubit $X$ or $Z$ plaquette, touching only nearest neighbors — just tiled over a bigger grid. Compare this with notebooks 04–07: to go from the Steane code's distance 3 to distance 5 would mean finding an entirely new classical code with different, generally non-local parity checks, then re-deriving its CSS structure from scratch. The surface code instead trades qubit count ($n$ growing as $d^2$, quadratically) for distance ($d$ growing linearly) using one unchanging, local, hardware-friendly check pattern — which is exactly why Section I of the reading frames this as the practical path to *large-scale* quantum computation, rather than an incremental improvement on a small code.

## Exercises

**1.** Write `count_qubits_touching_boundary(distance)` that, using `surface_code_layout`, counts how many data qubits lie on the two-qubit-wide edge of the grid (hint: use the `coords` dictionary's minimum/maximum $x$ or $y$ values) versus in the interior. How does the fraction of "boundary" data qubits change as `distance` grows — does the surface code become more or less efficient (qubits spent on data vs. structural overhead) at larger distance?

**2.** Section 4 injected a *single* hand-placed error into a 2-round circuit and looked at which detectors fired. Write `defect_count_distribution(distance, physical_error_rate, shots)` that uses Stim's built-in noise (via `after_clifford_depolarization=physical_error_rate` in `stim.Circuit.generated`) instead of a hand-placed error, samples the detector outcomes for many shots, and returns how many defects fired per shot, averaged over `shots`. Try it at a low and a high `physical_error_rate` — does the average defect count behave the way you'd expect?

In [7]:
def count_qubits_touching_boundary(distance):
    pass  # TODO: implement, then remove this stub body


def defect_count_distribution(distance, physical_error_rate, shots):
    pass  # TODO: implement, then remove this stub body


# TODO: once implemented, try e.g.:
# print(count_qubits_touching_boundary(5))
# print(defect_count_distribution(5, 0.01, 1000))
# print(defect_count_distribution(5, 0.2, 1000))